# European Renewable Electricity Analysis (2015–2025)

**Noah Nentwig** · Python, pandas, data validation, exploratory analysis and dashboard reporting.

This project examines recorded renewable electricity generation across the European countries represented in the supplied dataset. Generation growth alone does not establish a greener electricity mix.

**Execution status:** Prepared for GitHub from the submitted notebook. The per-capita conversion has been corrected before plotting and paths are relative to this repository. Saved outputs have been cleared because the two original CSVs were not supplied; this version has not been rerun. The presentation and dashboard PDF are original exports and retain their original titles.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb

## 1. Load the source data

Place the two original CSVs in `data/raw/`. Their provider, license and provenance still need to be documented before redistribution.


In [ ]:
from pathlib import Path

# Supports launching Jupyter from either the repository root or notebooks/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
required_files = [DATA_DIR / "renewable_energy_readings.csv", DATA_DIR / "countries.csv"]
missing_files = [str(path) for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError("Add the original source CSVs before running: " + ", ".join(missing_files))

df_1 = pd.read_csv(required_files[0])
df_2 = pd.read_csv(required_files[1])
for frame in (df_1, df_2):
    print(frame.shape)
    print(frame.dtypes)
    display(frame.head())
    display(frame.describe())


## 2. Inspect and clean the data

The submitted workflow removes duplicates, standardizes source and status labels, parses dates and flags suspect measurements. Deduplication currently occurs before category normalization; semantic duplicates require a further review.


In [ ]:
df_1.columns = df_1.columns.str.strip().str.lower().str.replace(" ", "_")

In [ ]:
print("Exact duplicates:", df_1.duplicated().sum())
columns_to_check = df_1.columns.drop("record_id")

print(
    "Duplicates excluding record_id:",
    df_1.duplicated(subset=columns_to_check).sum()
)
df_1 = df_1.drop_duplicates(
    subset=columns_to_check
).copy()

print("Shape after removing duplicates:", df_1.shape)

In [ ]:
df_1.head()

In [ ]:
df_1.isna().sum()

In [ ]:
print(df_1["energy_source"].value_counts())
print(df_1["data_status"].value_counts())

In [ ]:
df_1["energy_source"] = (
    df_1["energy_source"]
    .str.strip()
    .str.lower()
)

df_1["energy_source"] = df_1["energy_source"].replace({
    "solar pv": "solar",
    "bio-mass": "biomass",
    "hydro-power": "hydro"
})

df_1["data_status"] = (
    df_1["data_status"]
    .str.strip()
    .str.lower()
    .replace({
        "est": "estimated",
        "prov.": "provisional"
    })
)

print(df_1["energy_source"].value_counts())
print(df_1["data_status"].value_counts())

In [ ]:
df_1["period_date"] = pd.to_datetime(
    df_1["period_date"],
    errors="coerce"
)

df_1["last_updated"] = pd.to_datetime(
    df_1["last_updated"],
    errors="coerce"
)

print(df_1[["period_date", "last_updated"]].dtypes)

In [ ]:
print(
    "Negative generation values:",
    (df_1["generation_gwh"] < 0).sum()
)
print(
    "Share values above 100%:",
    (df_1["renewable_share_of_consumption_pct"] > 100).sum()
)

display(
    df_1[df_1["generation_gwh"] < 0].head(10)
)

In [ ]:
print("Missing values in df_1:")
print(df_1.isna().sum())
print("\nMissing values in df_2:")
print(df_2.isna().sum())
placeholders = ["", "na", "n/a", "nan", "none", "null", "-", "--", "?", "missing", "unknown"]
for df_name, df in (("df_1", df_1), ("df_2", df_2)):
    for col in df.select_dtypes(include=["object", "string"]).columns:
        found = df[col].astype(str).str.strip().str.lower().isin(placeholders).sum()
        if found:
            print(f"{df_name}.{col}: {found} placeholder values")

In [ ]:
display(df_1.nlargest(10, "generation_gwh")[
    ["country_id", "period_date", "energy_source", "generation_gwh", "national_consumption_gwh"]
])

In [ ]:
before = df_1.isna().sum()

df_1["negative_generation"] = df_1["generation_gwh"] < 0

df_1["share_above_100"] = (
    df_1["renewable_share_of_consumption_pct"] > 100
)

df_1["generation_above_consumption"] = (
    df_1["generation_gwh"] > df_1["national_consumption_gwh"]
)

print("Negative generation:", df_1["negative_generation"].sum())
print("Share above 100%:", df_1["share_above_100"].sum())
print(
    "Generation above consumption:",
    df_1["generation_above_consumption"].sum()
)

after = df_1.isna().sum()
print(pd.DataFrame({"NaN before": before, "NaN after": after, "added": after - before}))

## 3. Join country metadata and inspect reporting coverage

Country coverage varies across years. Presence in every year does not guarantee complete months or energy sources.


In [ ]:
df_2.columns = (
    df_2.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

df = pd.merge(
    df_1,
    df_2,
    on="country_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

print(df["_merge"].value_counts())

df = df.drop(columns="_merge")

display(df.head())
print(df.shape)

In [ ]:
df.head()

In [ ]:
yearly_coverage = (
    df.groupby("year")
    .agg(
        observations=("record_id", "count"),
        countries=("country_id", "nunique"),
        missing_generation=("generation_gwh", lambda x: x.isna().sum())
    )
    .reset_index()
)

display(yearly_coverage)

In [ ]:
plt.figure(figsize=(9, 5))

sb.countplot(
    data=df,
    x="energy_source",
    order=df["energy_source"].value_counts().index
)

plt.title("Number of Records by Renewable Energy Source")
plt.xlabel("Energy Source")
plt.ylabel("Number of Records")
plt.xticks(rotation=30)
plt.show()

In [ ]:
display(
    df.nlargest(15, "generation_gwh")[
        [
            "country_name",
            "year",
            "month",
            "energy_source",
            "generation_gwh",
            "national_consumption_gwh",
            "data_status"
        ]
    ]
)

## 4. Validate generation measurements

Negative readings and readings above the capacity-based monthly maximum are excluded from generation aggregates. Missing capacity prevents this validation for affected rows.


In [ ]:
df["hours_in_month"] = (
    df["period_date"].dt.days_in_month * 24
)


df["max_possible_generation_gwh"] = (
    df["installed_capacity_mw"] *
    df["hours_in_month"] / 1000
)


df["impossible_generation"] = (
    df["generation_gwh"] >
    df["max_possible_generation_gwh"]
)

print(
    "Physically impossible generation records:",
    df["impossible_generation"].sum()
)

display(
    df.loc[
        df["impossible_generation"],
        [
            "country_name",
            "year",
            "month",
            "energy_source",
            "generation_gwh",
            "installed_capacity_mw",
            "max_possible_generation_gwh"
        ]
    ].head(10)
)

In [ ]:
df_analysis = df.copy()


invalid_generation = (
    df_analysis["negative_generation"] |
    df_analysis["impossible_generation"]
)

df_analysis.loc[
    invalid_generation,
    "generation_gwh"
] = np.nan

print("Original dataset:", df.shape)
print("Analysis dataset:", df_analysis.shape)

print(
    "Valid generation measurements:",
    df_analysis["generation_gwh"].notna().sum()
)

## 5. Recorded generation and technology trends

The 2015–2025 totals reflect changing reporting coverage. The 2018–2025 cohort comparison holds country membership constant but does not guarantee complete source-month coverage.


In [ ]:
yearly_generation = (
    df_analysis.groupby("year")["generation_gwh"]
    .sum(min_count=1)
    .reset_index()
)

display(yearly_generation)

plt.figure(figsize=(10, 5))

sb.lineplot(
    data=yearly_generation,
    x="year",
    y="generation_gwh",
    marker="o"
)

plt.title("Renewable Electricity Generation in Europe (2015–2025)")
plt.xlabel("Year")
plt.ylabel("Total Renewable Generation (GWh)")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
df_recent = df_analysis[
    df_analysis["year"].between(2018, 2025)
].copy()


country_years = (
    df_recent.groupby("country_id")["year"]
    .nunique()
)

common_countries = country_years[
    country_years == 8
].index

print(
    "Countries represented in all 8 years:",
    len(common_countries)
)


df_consistent = df_recent[
    df_recent["country_id"].isin(common_countries)
].copy()

print(
    df_consistent.groupby("year")["country_id"].nunique()
)

In [ ]:
coverage_check = (
    df_consistent.groupby("year")
    .agg(
        total_records=("record_id", "count"),
        valid_generation=("generation_gwh", "count"),
        countries=("country_id", "nunique")
    )
    .reset_index()
)

coverage_check["valid_pct"] = (
    coverage_check["valid_generation"]
    / coverage_check["total_records"] * 100
)

display(coverage_check)

In [ ]:
source_trends = (
    df_consistent.groupby(
        ["year", "energy_source"]
    )["generation_gwh"]
    .sum(min_count=1)
    .reset_index()
)

plt.figure(figsize=(11, 6))

sb.lineplot(
    data=source_trends,
    x="year",
    y="generation_gwh",
    hue="energy_source",
    marker="o"
)

plt.title(
    "Renewable Electricity Generation by Source (2018–2025)"
)
plt.xlabel("Year")
plt.ylabel("Recorded Generation (GWh)")
plt.legend(title="Energy Source")
plt.grid(alpha=0.3)
plt.show()

## 6. Country comparisons in 2025

Absolute and per-person generation answer different questions. Population is supplied metadata; verify its reference year before interpreting per-capita comparisons.


In [ ]:
df_2025 = df_consistent[
    df_consistent["year"] == 2025
].copy()


country_generation = (
    df_2025.groupby("country_name")
    .agg(
        total_generation_gwh=("generation_gwh", "sum"),
        valid_records=("generation_gwh", "count"),
        total_records=("record_id", "count")
    )
    .reset_index()
)

country_generation["valid_pct"] = (
    country_generation["valid_records"]
    / country_generation["total_records"] * 100
)


country_generation = country_generation.sort_values(
    "total_generation_gwh",
    ascending=False
)

display(country_generation.head(10))

In [ ]:
top_10 = country_generation.head(10)

plt.figure(figsize=(11, 6))

sb.barplot(
    data=top_10,
    x="total_generation_gwh",
    y="country_name",
    color="steelblue"
)

plt.title("Top 10 Countries by Recorded Renewable Generation (2025)")
plt.xlabel("Renewable Generation (GWh)")
plt.ylabel("Country")
plt.tight_layout()
plt.show()

In [ ]:
country_coverage = (
    df_2025.groupby(["country_name", "energy_source"])
    .agg(
        months_recorded=("month", "nunique"),
        valid_readings=("generation_gwh", "count")
    )
    .reset_index()
)

display(
    country_coverage.sort_values(
        ["country_name", "energy_source"]
    ).head(20)
)

### Generation per person

Convert GWh to MWh and population in millions to people: `GWh / (population_millions × 1,000)` gives MWh/person. The correction is applied before sorting and plotting.


In [ ]:
country_per_capita = (
    df_2025.groupby(
        ["country_name", "population_millions"],
        as_index=False
    )["generation_gwh"]
    .sum(min_count=1)
)


country_per_capita["mwh_per_person"] = (
    country_per_capita["generation_gwh"]
    / (country_per_capita["population_millions"] * 1000)
)

country_per_capita = country_per_capita.sort_values(
    "mwh_per_person",
    ascending=False
)

display(country_per_capita.head(10))

In [ ]:
top_10_per_capita = country_per_capita.head(10)

plt.figure(figsize=(11, 6))

sb.barplot(
    data=top_10_per_capita,
    x="mwh_per_person",
    y="country_name",
    color="seagreen"
)

plt.title("Top 10 Countries: Recorded Renewable Generation per Person (2025)")
plt.xlabel("Renewable Generation (MWh per Person)")
plt.ylabel("Country")
plt.tight_layout()
plt.show()

## 7. Supplied 2030 target metadata

These values and their definitions have not been verified against official policy sources. This chart describes supplied metadata, not confirmed national commitments or progress toward targets.


In [ ]:
targets = (
    df[[
        "country_name",
        "target_renewable_share_2030_pct"
    ]]
    .drop_duplicates(subset="country_name")
    .dropna()
    .sort_values(
        "target_renewable_share_2030_pct",
        ascending=False
    )
)

display(targets.head(10))

plt.figure(figsize=(11, 6))

sb.barplot(
    data=targets.head(10),
    x="target_renewable_share_2030_pct",
    y="country_name",
    color="mediumpurple"
)

plt.title("Top 10 Highest Renewable Energy Targets for 2030")
plt.xlabel("2030 Renewable Share Target (%)")
plt.ylabel("Country")
plt.tight_layout()
plt.show()